In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import spikeinterface as si
from spikeinterface.preprocessing import notch_filter
from mne_connectivity import spectral_connectivity_epochs
import sys 

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))

EPHYS_DIR = REPO_ROOT / "DATA" / "ephys"
DATA_DIR = REPO_ROOT / "DATA"
ANALYSIS_WINDOWS_FILE = DATA_DIR / "analysis_windows.csv"

GRANGER_FMIN_HZ = 1
GRANGER_FMAX_HZ = 101
GRANGER_MODE = "multitaper"
GRANGER_N_JOBS = 1
GRANGER_FREQUENCY_BINS = np.arange(1, 101)

NOTCH_FREQUENCY_HZ = 50
NOTCH_Q = 35

In [ ]:
# Load and validate the synchronized analysis windows used for Granger analysis

analysis_windows = pd.read_csv(
    ANALYSIS_WINDOWS_FILE
)

required_columns = {
    "recording",
    "label",
    "sample_start",
    "sample_end",
}

missing_columns = required_columns - set(
    analysis_windows.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print(
    f"Loaded {len(analysis_windows)} analysis windows."
)

recording_dirs = sorted(
    {
        path.parent.parent
        for path in EPHYS_DIR.rglob("lfp/meta.json")
    }
)

print(
    f"Found {len(recording_dirs)} recordings."
)



In [ ]:
# Compute directional PFC-RSC Granger causality for each behavioral condition

granger_results = []

for recording_dir in recording_dirs:

    recording_name = recording_dir.name

    print(f"\nProcessing: {recording_name}")

    lfp_path = recording_dir / "lfp"

    recording = si.load(lfp_path)

    sampling_frequency = float(
        recording.get_sampling_frequency()
    )

    with (lfp_path / "meta.json").open(
        "r",
        encoding="utf-8",
    ) as file:
        metadata = json.load(file)

    pfc_channels = metadata["anatomical_assignment"]["pfc_channels"]
    rsc_channels = metadata["anatomical_assignment"]["rsc_channels"]

    traces = recording.get_traces()

    lfp_data = pd.DataFrame(
        traces,
        columns=recording.channel_ids,
    )

    regional_lfp = pd.DataFrame(
        {
            "PFC": lfp_data[pfc_channels].mean(axis=1),
            "RSC": lfp_data[rsc_channels].mean(axis=1),
        }
    )

    regional_recording = si.NumpyRecording(
        regional_lfp.values,
        sampling_frequency=sampling_frequency,
        channel_ids=["PFC", "RSC"],
    )

    filtered_recording = notch_filter(
        regional_recording,
        freq=NOTCH_FREQUENCY_HZ,
        q=NOTCH_Q,
    )

    recording_windows = analysis_windows[
        analysis_windows["recording"] == recording_name
    ]

    for label, condition_windows in recording_windows.groupby(
        "label"
    ):

        granger_epochs = []

        for _, window in condition_windows.iterrows():

            sample_start = int(
                window["sample_start"]
            )

            sample_end = int(
                window["sample_end"]
            )

            if sample_end <= sample_start:
                continue

            lfp_segment = filtered_recording.get_traces(
                start_frame=sample_start,
                end_frame=sample_end,
                channel_ids=["PFC", "RSC"],
            )

            if len(lfp_segment) == 0:
                continue

            granger_epochs.append(
                lfp_segment.T
            )

        if len(granger_epochs) < 2:
            continue

        granger_epochs = np.stack(
            granger_epochs,
            axis=0,
        )

        indices = (
            [[0], [1]],
            [[1], [0]],
        )

        granger = spectral_connectivity_epochs(
            granger_epochs,
            method="gc",
            mode=GRANGER_MODE,
            sfreq=sampling_frequency,
            fmin=GRANGER_FMIN_HZ,
            fmax=GRANGER_FMAX_HZ,
            indices=indices,
            faverage=False,
            n_jobs=GRANGER_N_JOBS,
            verbose=False,
        )

        granger_values = granger.get_data()
        granger_frequencies = granger.freqs

        directions = [
            ("PFC", "RSC"),
            ("RSC", "PFC"),
        ]

        for direction_index, (
            source,
            target,
        ) in enumerate(directions):

            for frequency_bin in GRANGER_FREQUENCY_BINS:

                frequency_mask = (
                    (granger_frequencies >= frequency_bin)
                    & (
                        granger_frequencies
                        < frequency_bin + 1
                    )
                )

                if not frequency_mask.any():
                    continue

                granger_value = granger_values[
                    direction_index,
                    frequency_mask,
                ].mean()

                granger_results.append(
                    {
                        "recording": recording_name,
                        "label": label,
                        "source": source,
                        "target": target,
                        "frequency_hz": int(
                            frequency_bin
                        ),
                        "granger": float(
                            granger_value
                        ),
                    }
                )

In [ ]:
# Assemble and save the Granger causality results

granger_df = pd.DataFrame(
    granger_results
)

if granger_df.empty:
    raise ValueError(
        "No Granger results were generated."
    )

granger_df = granger_df.sort_values(
    [
        "recording",
        "label",
        "source",
        "target",
        "frequency_hz",
    ]
).reset_index(drop=True)

print(
    f"Generated {len(granger_df)} rows."
)

display(granger_df.head())

OUTPUT_FILE = DATA_DIR / "granger_results.csv"

granger_df.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Saved: {OUTPUT_FILE}")